# 07 — Outliers + Correlación — Airbnb Listings

## Setup

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

try:
    from scipy import stats
    SCIPY = True
except ImportError:
    SCIPY = False
    print('scipy no instalado — se usa implementación manual del z-score')

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT = find_project_root()
df = pd.read_csv(ROOT / 'data' / 'external' / 'Listings.csv',
                 encoding='latin-1', low_memory=False)

# Tipos correctos antes de cualquier análisis
df['host_since'] = pd.to_datetime(df['host_since'])
for col in ['host_is_superhost', 'host_has_profile_pic',
            'host_identity_verified', 'instant_bookable']:
    df[col] = df[col].map({'t': True, 'f': False})

print(f'Shape: {df.shape}')
print(f'price — min: {df["price"].min()}, max: {df["price"].max():,}, media: {df["price"].mean():.0f}')


scipy no instalado — se usa implementación manual del z-score
Shape: (279712, 33)
price — min: 0, max: 625,216, media: 609


---
# Bloque 1 — Outliers

Un outlier es un valor que se aleja significativamente del resto. Puede ser un error de datos, un caso real extremo, o algo representativo del negocio. El método detecta — la decisión es contextual.

## Diagnóstico previo — distribución de `price`

Antes de aplicar ningún método, ver cómo está distribuida la columna. La forma de la distribución determina qué método es más adecuado.

In [2]:
print(df['price'].describe().round(2))
print(f'\nAsimetría (skewness): {df["price"].skew():.2f}')
# Skewness > 1 o < -1 indica distribución asimétrica
# En ese caso IQR es más robusto que z-score
print(f'Valores en 0: {(df["price"] == 0).sum()}')
# Un precio de 0 es casi siempre un error de datos


count    279712.00
mean        608.79
std        3441.83
min           0.00
25%          75.00
50%         150.00
75%         474.00
max      625216.00
Name: price, dtype: float64

Asimetría (skewness): 61.62
Valores en 0: 113


In [ ]:
fig = px.histogram(df[df['price'] < 1000], x='price', nbins=80,
                   title='Distribución de precio por noche (excluye >1000 para visibilidad)',
                   labels={'price': 'Precio (USD/noche)'})
fig.show()


## Método 1 — IQR

Mejor para distribuciones asimétricas (la mayoría de variables de negocio). No asume ninguna forma de distribución. Los límites se calculan desde los cuartiles, no desde la media.

In [ ]:
Q1  = df['price'].quantile(0.25)
Q3  = df['price'].quantile(0.75)
IQR = Q3 - Q1

# El factor 1.5 es el estándar de Tukey — cubre ~99.3% de una distribución normal
# Con 3.0 se detectan solo outliers extremos
limite_inf = Q1 - 1.5 * IQR
limite_sup = Q3 + 1.5 * IQR

outliers_iqr = df[(df['price'] < limite_inf) | (df['price'] > limite_sup)]

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}')
print(f'Límite inferior: {limite_inf:.0f}')
print(f'Límite superior: {limite_sup:.0f}')
print(f'\nOutliers IQR: {len(outliers_iqr):,} filas ({len(outliers_iqr)/len(df)*100:.1f}%)')


## Método 2 — Z-score

Mide cuántas desviaciones estándar se aleja un valor de la media. El umbral estándar es 3 (cubre el 99.7% de una distribución normal). Menos robusto cuando la distribución es asimétrica porque la media y la desviación estándar se distorsionan con los propios outliers.

In [ ]:
if SCIPY:
    df['zscore_price'] = stats.zscore(df['price'].dropna())
else:
    # Implementación manual: (valor - media) / desviación estándar
    media = df['price'].mean()
    std   = df['price'].std()
    df['zscore_price'] = (df['price'] - media) / std

outliers_z = df[df['zscore_price'].abs() > 3]

print(f'Outliers z-score (|z| > 3): {len(outliers_z):,} filas ({len(outliers_z)/len(df)*100:.1f}%)')
print(f'\nComparación:')
print(f'  IQR detectó:     {len(outliers_iqr):>6,} outliers')
print(f'  Z-score detectó: {len(outliers_z):>6,} outliers')

# Los dos métodos casi nunca dan el mismo número
# IQR suele detectar más en distribuciones asimétricas


## Visualizar: boxplot + distribución

In [ ]:
fig = go.Figure()

fig.add_trace(go.Box(
    y=df['price'],
    name='price',
    boxpoints='outliers',   # muestra solo los puntos outlier, no todos
    marker_color='steelblue',
    line_color='steelblue'
))

# Líneas de límites IQR
for valor, label in [(limite_sup, 'Límite IQR sup'), (limite_inf, 'Límite IQR inf')]:
    fig.add_hline(y=valor, line_dash='dash', line_color='tomato',
                  annotation_text=label, annotation_position='right')

fig.update_layout(title='Boxplot de precio por noche — límites IQR marcados',
                  yaxis_title='Precio (USD)')
fig.show()


## `minimum_nights` — outliers con decisión contextual

Esta columna tiene outliers legítimos (alquileres de larga estancia) y posibles errores (valores extremos como 365, 1125). El contexto de negocio es lo que determina qué hacer con cada caso.

In [ ]:
print(df['minimum_nights'].describe().round(1))
print(f'\nDistribución de valores extremos:')
print(df['minimum_nights'].value_counts().head(10))


In [ ]:
Q1_mn  = df['minimum_nights'].quantile(0.25)
Q3_mn  = df['minimum_nights'].quantile(0.75)
IQR_mn = Q3_mn - Q1_mn
lim_mn = Q3_mn + 1.5 * IQR_mn

extremos = df[df['minimum_nights'] > lim_mn]
print(f'Límite IQR superior: {lim_mn:.0f} noches')
print(f'Outliers detectados: {len(extremos):,} ({len(extremos)/len(df)*100:.1f}%)')
print()
print('Decisión según contexto:')
print('  minimum_nights == 1125 → probable error de entrada de datos → eliminar')
print('  minimum_nights == 365  → alquiler anual legítimo → conservar y anotar')
print('  minimum_nights == 30   → alquiler mensual → conservar, puede ser subgrupo')
print()
# Mostrar cuántos tienen valores > 365 (claramente erróneos)
print(f'Listados con >365 noches mínimas: {(df["minimum_nights"] > 365).sum()}')


---
# Bloque 2 — Correlación

La correlación mide la relación lineal entre dos variables numéricas. Va de -1 a 1.
- `1.0` — relación perfecta positiva (cuando una sube, la otra sube igual)
- `-1.0` — relación perfecta negativa (cuando una sube, la otra baja igual)
- `0.0` — sin relación lineal

Correlación alta entre dos variables en un modelo indica que están aportando la misma información.

## Selección de columnas numéricas relevantes

In [ ]:
cols_num = [
    'price',
    'accommodates',
    'bedrooms',
    'minimum_nights',
    'review_scores_rating',
    'review_scores_cleanliness',
    'host_total_listings_count',
]

# Solo las columnas que existen en el DataFrame
cols_num = [c for c in cols_num if c in df.columns]

df_num = df[cols_num].dropna()
print(f'Filas con todas las columnas completas: {len(df_num):,}')
print(df_num.describe().round(2))


## Matriz de correlación

In [ ]:
corr = df_num.corr().round(2)
print(corr)


## Heatmap — máscara triangular

La matriz de correlación es simétrica: el valor de A↔B es el mismo que B↔A. Se muestra solo el triángulo inferior para evitar la redundancia visual.

In [ ]:
# Máscara: poner NaN en el triángulo superior para que no se muestre
corr_masked = corr.copy().astype(float)
for i in range(len(corr_masked)):
    for j in range(i + 1, len(corr_masked.columns)):
        corr_masked.iloc[i, j] = float('nan')

fig = px.imshow(
    corr_masked,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',   # rojo = negativo, azul = positivo
    zmin=-1, zmax=1,
    title='Correlación entre métricas de listing'
)
fig.update_layout(coloraxis_colorbar_title='r')
fig.show()


## Interpretar los resultados

In [ ]:
print('Pares con correlación alta (|r| > 0.5):')
print('─' * 45)

# Iterar solo el triángulo inferior para no duplicar pares
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) > 0.5:
            col_a = corr.columns[i]
            col_b = corr.columns[j]
            direccion = 'positiva' if r > 0 else 'negativa'
            print(f'{col_a} ↔ {col_b}: r={r:.2f} ({direccion})')

print()
print('Pares sin relación (|r| < 0.1):')
print('─' * 45)
for i in range(len(corr.columns)):
    for j in range(i):
        r = corr.iloc[i, j]
        if abs(r) < 0.1:
            print(f'{corr.columns[i]} ↔ {corr.columns[j]}: r={r:.2f}')


## Correlación ≠ causalidad

`r` mide relación lineal, no causa. Dos cosas pueden correlacionar por una tercera variable que no está en el análisis.

| r | Interpretación habitual |
|---|-------------------------|
| 0.9 – 1.0 | Muy alta — probable redundancia entre variables |
| 0.7 – 0.9 | Alta |
| 0.5 – 0.7 | Moderada |
| 0.3 – 0.5 | Débil |
| < 0.3 | Sin relación lineal relevante |

En modelado: si dos features tienen `r > 0.85`, eliminar una de las dos antes de entrenar.

---
## Resumen de decisiones sobre outliers

| Caso | Decisión | Razón |
|------|----------|-------|
| Precio = 0 | Eliminar | Error de datos — ningún listing es gratuito |
| Precio = 50,000 | Investigar | Puede ser un error o un alquiler de lujo real |
| minimum_nights > 365 | Eliminar | Imposible alquilar más noches de las que tiene un año |
| minimum_nights = 365 | Conservar + anotar | Alquiler anual legítimo, subgrupo distinto |
| review_score = 1.0 | Conservar | Puntuación válida aunque extrema |
